In [2]:
import pandas as pd
import glob
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import openai
from bertopic.representation import OpenAI

from hdbscan import HDBSCAN

In [3]:
df_news = pd.read_parquet('./Data/all_news_1984_2024.parquet')
df_news['Date'] = pd.to_datetime(df_news['Date'], format='%Y-%m-%d')

In [6]:
year = 1985

df_gpr = pd.read_csv(f"./Result/GPR_Peaks/data/gpr_peaks_{year}.csv")
filtered_news = df_news[df_news['Date'].isin(df_gpr['date'])]
filtered_news

,Date,Caption,Content
43420,1985-01-14,Pepper...and Salt: [2],Winter Shuffle It's the season of Sun Belt vac...
43421,1985-01-14,Pentagon's Authority to Get Audit Data From Co...,PITTSBURGH -- The federal government is toughe...
43422,1985-01-14,Pentagon Gets Right to Review Licensing Of Hig...,WASHINGTON -- President Reagan authorized the ...
43423,1985-01-14,Pennzoil Co.,"Pennzoil Co. (Houston) -- Norman J. Luke, grou..."
43424,1985-01-14,The Outlook: Interest Rates Seem Likely to Ris...,NEW YORK -- Last week's game of musical chairs...
...,...,...,...
80210,1985-12-31,Eurobonds in Marks Totaling $2.3 Billion Readi...,LONDON -- News that 23 borrowers are scheduled...
80211,1985-12-31,"Exchanges, Banks to Close Tomorrow for New Yea...","Stock and commodity exchanges, banks and most ..."
80212,1985-12-31,"Executives' Expectations For Quarter Low, Poll...",NEW YORK -- Dun & Bradstreet Corp. said busine...
80213,1985-12-31,Financing Business: Diversified Industries Inc.,DIVERSIFIED INDUSTRIES INC. said it entered in...


In [12]:
for year in range(1985, 2025):
    # df_gpr = pd.read_csv(f"./Result/GPR_Peaks/data/gpr_peaks_{year}.csv")
    # filtered_news = df_news[df_news['Date'].isin(df_gpr['date'])]
    filtered_news = df_news[df_news['Date'].dt.year == year]
    print(year, len(filtered_news))

1985 37959
1986 35095
1987 38052
1988 36270
1989 44887
1990 56466
1991 54037
1992 65207
1993 59570
1994 86310
1995 57407
1996 44840
1997 75774
1998 43580
1999 45157
2000 40472
2001 34935
2002 32188
2003 34311
2004 37382
2005 37564
2006 30775
2007 2
2008 37407
2009 44268
2010 76178
2011 52632
2012 38660
2013 37409
2014 35628
2015 33628
2016 32492
2017 31734
2018 27612
2019 26820
2020 26900
2021 16397
2022 21173
2023 24371
2024 11834


In [28]:
docs = filtered_news['Content'].tolist()
timestamps = filtered_news['Date'].tolist()

# hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model, 
                    #    min_topic_size=11,
                        nr_topics=35,
                       verbose=True)
topics, probs = topic_model.fit_transform(docs)
topic_info = topic_model.get_topic_info()
print("Topic Info:")
print(topic_info)
topic_model.visualize_topics(width=1000, height=1000)

2025-06-29 19:06:19,974 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

2025-06-29 19:06:23,019 - BERTopic - Embedding - Completed ✓
2025-06-29 19:06:23,019 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-06-29 19:06:27,105 - BERTopic - Dimensionality - Completed ✓
2025-06-29 19:06:27,106 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-06-29 19:06:27,145 - BERTopic - Cluster - Completed ✓
2025-06-29 19:06:27,145 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2025-06-29 19:06:27,342 - BERTopic - Representation - Completed ✓
2025-06-29 19:06:27,343 - BERTopic - Topic reduction - Reducing number of topics
2025-06-29 19:06:27,347 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-06-29 19:06:27,533 - BERTopic - Representation - Completed ✓
2025-06-29 19:06:27,534 - BERTopic - Topic reduction - Reduced number of topics from 58 to 35


Topic Info:
    Topic  Count                                          Name  \
0      -1    892                              -1_the_to_of_and   
1       0    253                       0_israel_hamas_gaza_the   
2       1    157                                1_the_to_in_of   
3       2    136                              2_the_gas_oil_to   
4       3    136                             3_mr_the_trump_he   
5       4    123                             4_china_the_to_in   
6       5    121                              5_the_to_that_of   
7       6    108                  6_ukraine_russia_the_russian   
8       7    107                              7_the_game_to_of   
9       8     94                                8_ai_to_and_of   
10      9     93           9_commentary_analysis_financial_and   
11     10     83                       10_students_of_the_that   
12     11     70                            11_drug_the_and_to   
13     12     69                            12_trump_the_in_to  

In [ ]:
# topic_model.reduce_topics(docs, nr_topics=20)
# topic_model.visualize_topics(width=500, height=500)

In [29]:
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_OPENAI_API_KEY")

representation_model = OpenAI(
    client, model='gpt-4o' ,exponential_backoff=True, chat=True, prompt=prompt
)

topic_model.update_topics(docs, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

df_result = topic_model.get_topic_info()
df_result.to_csv('result_2.csv')

100%|██████████| 35/35 [00:40<00:00,  1.16s/it]


In [7]:
df_result

,Topic,Count,Name,Representation,Representative_Docs
0,-1,949,-1_Global Economic Uncertainty and Financial S...,[Global Economic Uncertainty and Financial Sec...,[Wall Street's traders and deal makers are str...
1,0,164,0_Political and Security Challenges in Israel ...,[Political and Security Challenges in Israel a...,[Palestine 1936 By Oren Kessler Rowman & Littl...
2,1,104,1_Sports Legends and Major Tournament Wins,[Sports Legends and Major Tournament Wins],[Brooks Koepka didn't sleep the Sunday night a...
3,2,104,2_2024 U.S. Presidential Election and Party Dy...,[2024 U.S. Presidential Election and Party Dyn...,"[This is the season of Democratic discontent, ..."
4,3,93,3_Global Energy Market Dynamics and Geopolitics,[Global Energy Market Dynamics and Geopolitics],[Russia said it plans to cut oil production by...
5,4,93,4_Financial Commentary and Analysis,[Financial Commentary and Analysis],"[[Financial Analysis and Commentary], [Financi..."
6,5,92,5_Ukraine-Russia War and International Diploma...,[Ukraine-Russia War and International Diplomat...,"[VILNIUS, Lithuania -- French President Emmanu..."
7,6,76,6_Financial Challenges and Strategic Shifts in...,[Financial Challenges and Strategic Shifts in ...,[Goldman Sachs Group Inc. Chief Executive Davi...
8,7,65,7_Economic Policy and Student Loan Reform,[Economic Policy and Student Loan Reform],[WASHINGTON -- An era of ultracheap debt is ov...
9,8,64,8_AI Impact and Ethical Considerations,[AI Impact and Ethical Considerations],[Seeing the new artificial intelligence-powere...


In [8]:
topic_model.visualize_topics(width=500, height=500)